In [ ]:
!pip install -q tensorflow pandas numpy scikit-learn

In [ ]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
%%writefile train_spendsense.py


"""
============================================================
SpendSense Transaction Categorization Model
============================================================

Dataset Schema:
    transaction_text  -> Model Input
    merchant          -> Output Head 1
    category          -> Output Head 2
    category_id       -> Metadata only (not used for training)
    augmentation_type -> Metadata only (not used for training)

Architecture:
    Character-level TextVectorization
            ↓
    Embedding
            ↓
    Multi-Kernel CNN (3, 5, 7)
            ↓
    Global Max Pooling
            ↓
    Shared Dense Representation
            ↓
       ┌────┴─────┐
       ↓          ↓
    Merchant   Category
    Softmax    Softmax

Outputs:
    - spendsense_model.keras
    - spendsense_float32.tflite
    - spendsense_float16.tflite
    - merchant_labels.json
    - category_labels.json
    - evaluation_results.json
============================================================
"""

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

# Dataset path
DATA_PATH = "/content/indian_payment_transaction_dataset (1).csv"

# Output directory
MODEL_DIR = "/content/spendsense_model_output"

# Text vectorization
MAX_VOCAB_SIZE = 5000
MAX_SEQUENCE_LENGTH = 128

# Model architecture
EMBEDDING_DIM = 64
CNN_FILTERS = 128
DENSE_UNITS = 128
DROPOUT_RATE = 0.30

# Training
BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 1e-3

VALIDATION_SIZE = 0.10
TEST_SIZE = 0.10


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):
    """Set random seeds."""

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    print(f"✓ Random seed set to {seed}")


# ============================================================
# LOAD DATASET
# ============================================================

def load_dataset():

    print("\n" + "=" * 70)
    print("LOADING DATASET")
    print("=" * 70)

    if not os.path.exists(DATA_PATH):
        print(f"\nERROR: Dataset not found at:\n{DATA_PATH}")

        print("\nFiles available in /content:")
        for f in os.listdir("/content"):
            print(" -", f)

        raise FileNotFoundError(DATA_PATH)

    df = pd.read_csv(DATA_PATH)

    print(f"\nDataset Shape: {df.shape}")

    print("\nDataset Columns:")
    print(df.columns.tolist())

    # --------------------------------------------------------
    # Validate required columns
    # --------------------------------------------------------

    required_columns = [
        "transaction_text",
        "merchant",
        "category",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"\nMissing required columns: {missing_columns}"
        )

    # Keep only required columns for training
    df = df[required_columns].copy()

    print("\n✓ Required columns validated")

    # --------------------------------------------------------
    # Clean transaction text
    # --------------------------------------------------------

    df["transaction_text"] = (
        df["transaction_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # --------------------------------------------------------
    # Clean labels
    # --------------------------------------------------------

    df["merchant"] = (
        df["merchant"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

    df["category"] = (
        df["category"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Remove empty transaction text
    # --------------------------------------------------------

    before = len(df)

    df = df[
        df["transaction_text"].str.len() > 0
    ].copy()

    print(
        f"Removed empty transaction rows: "
        f"{before - len(df)}"
    )

    # --------------------------------------------------------
    # Remove exact duplicates
    # --------------------------------------------------------

    before_duplicates = len(df)

    df = df.drop_duplicates().reset_index(drop=True)

    print(
        f"Removed exact duplicates: "
        f"{before_duplicates - len(df)}"
    )

    # Shuffle
    df = df.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    print(f"\nFinal Dataset Size: {len(df):,}")

    return df


# ============================================================
# DATASET VALIDATION
# ============================================================

def validate_dataset(df):

    print("\n" + "=" * 70)
    print("DATASET VALIDATION")
    print("=" * 70)

    print("\nMerchant Statistics:")
    print(
        f"Unique merchants: "
        f"{df['merchant'].nunique()}"
    )

    merchant_counts = (
        df["merchant"]
        .value_counts()
    )

    print(
        f"Minimum samples per merchant: "
        f"{merchant_counts.min()}"
    )

    print(
        f"Maximum samples per merchant: "
        f"{merchant_counts.max()}"
    )

    print("\nCategory Statistics:")
    print(
        f"Unique categories: "
        f"{df['category'].nunique()}"
    )

    print("\nCategory Distribution:")

    print(
        df["category"]
        .value_counts()
        .to_string()
    )

    # --------------------------------------------------------
    # Check transaction text conflicts
    # --------------------------------------------------------

    print("\nChecking label conflicts...")

    merchant_conflicts = (
        df.groupby("transaction_text")["merchant"]
        .nunique()
    )

    merchant_conflicts = (
        merchant_conflicts[
            merchant_conflicts > 1
        ]
    )

    category_conflicts = (
        df.groupby("transaction_text")["category"]
        .nunique()
    )

    category_conflicts = (
        category_conflicts[
            category_conflicts > 1
        ]
    )

    print(
        f"Same text → different merchant conflicts: "
        f"{len(merchant_conflicts)}"
    )

    print(
        f"Same text → different category conflicts: "
        f"{len(category_conflicts)}"
    )

    if len(category_conflicts) > 0:
        print(
            "\n⚠ WARNING: Same transaction_text has "
            "multiple category labels."
        )

    print("\n✓ Dataset validation complete")


# ============================================================
# CREATE LABEL ENCODERS
# ============================================================

def create_label_encoders(df, output_dir):

    print("\n" + "=" * 70)
    print("CREATING LABEL ENCODERS")
    print("=" * 70)

    output_dir = Path(output_dir)

    label_dir = output_dir / "labels"

    label_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    encoders = {}

    for task in ["merchant", "category"]:

        encoder = LabelEncoder()

        encoder.fit(
            df[task].astype(str)
        )

        encoders[task] = encoder

        labels = encoder.classes_.tolist()

        label_file = (
            label_dir /
            f"{task}_labels.json"
        )

        with open(
            label_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                labels,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\n{task.upper()}"
        )

        print(
            f"Number of classes: "
            f"{len(labels)}"
        )

        print(
            f"✓ Labels saved: "
            f"{label_file}"
        )

    return encoders


# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

def split_dataset(df):

    print("\n" + "=" * 70)
    print("DATASET SPLIT")
    print("=" * 70)

    # Stratify using category because
    # category is the primary classification target

    train_df, temp_df = train_test_split(
        df,
        test_size=VALIDATION_SIZE + TEST_SIZE,
        random_state=SEED,
        stratify=df["category"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=SEED,
        stratify=temp_df["category"]
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(
        f"\nTrain:      {len(train_df):,}"
    )

    print(
        f"Validation: {len(val_df):,}"
    )

    print(
        f"Test:       {len(test_df):,}"
    )

    return train_df, val_df, test_df


# ============================================================
# TEXT VECTORIZER
# ============================================================

def create_vectorizer(train_texts, output_dir):

    print("\n" + "=" * 70)
    print("CREATING CHARACTER VECTORIZER")
    print("=" * 70)

    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=MAX_VOCAB_SIZE,
        output_mode="int",
        output_sequence_length=MAX_SEQUENCE_LENGTH,
        split="character",
        standardize="lower_and_strip_punctuation",
        name="text_vectorizer"
    )

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(train_texts)
        .batch(1024)
    )

    vectorizer.adapt(dataset)

    vocabulary = vectorizer.get_vocabulary()

    print(
        f"\nVocabulary size: "
        f"{len(vocabulary)}"
    )

    # Save vocabulary
    vocab_path = (
        Path(output_dir) /
        "vocabulary.json"
    )

    with open(
        vocab_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            vocabulary,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"✓ Vocabulary saved: {vocab_path}"
    )

    return vectorizer


# ============================================================
# PREPARE TARGETS
# ============================================================

def prepare_targets(df, encoders):

    targets = {}

    for task in ["merchant", "category"]:

        targets[task] = (
            encoders[task]
            .transform(
                df[task].astype(str)
            )
            .astype(np.int32)
        )

    return targets


# ============================================================
# BUILD MODEL
# ============================================================

def build_model(vectorizer, encoders):

    print("\n" + "=" * 70)
    print("BUILDING MODEL")
    print("=" * 70)

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    text_input = tf.keras.Input(
        shape=(1,),
        dtype=tf.string,
        name="transaction_text"
    )

    # --------------------------------------------------------
    # Vectorization
    # --------------------------------------------------------

    x = vectorizer(text_input)

    vocab_size = len(
        vectorizer.get_vocabulary()
    )

    # --------------------------------------------------------
    # Embedding
    # --------------------------------------------------------

    x = tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        name="embedding"
    )(x)

    # --------------------------------------------------------
    # Multi-kernel CNN
    # --------------------------------------------------------

    branches = []

    for kernel_size in [3, 5, 7]:

        branch = tf.keras.layers.Conv1D(
            filters=CNN_FILTERS,
            kernel_size=kernel_size,
            padding="same",
            activation="relu",
            name=f"conv_{kernel_size}"
        )(x)

        branch = tf.keras.layers.GlobalMaxPooling1D(
            name=f"global_pool_{kernel_size}"
        )(branch)

        branches.append(branch)

    # --------------------------------------------------------
    # Merge CNN branches
    # --------------------------------------------------------

    x = tf.keras.layers.Concatenate(
        name="cnn_features"
    )(branches)

    # --------------------------------------------------------
    # Shared representation
    # --------------------------------------------------------

    x = tf.keras.layers.Dense(
        DENSE_UNITS,
        activation="relu",
        name="shared_dense"
    )(x)

    x = tf.keras.layers.Dropout(
        DROPOUT_RATE,
        name="dropout"
    )(x)

    # --------------------------------------------------------
    # Merchant head
    # --------------------------------------------------------

    merchant_classes = len(
        encoders["merchant"].classes_
    )

    merchant_output = tf.keras.layers.Dense(
        merchant_classes,
        activation="softmax",
        name="merchant"
    )(x)

    # --------------------------------------------------------
    # Category head
    # --------------------------------------------------------

    category_classes = len(
        encoders["category"].classes_
    )

    category_output = tf.keras.layers.Dense(
        category_classes,
        activation="softmax",
        name="category"
    )(x)

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = tf.keras.Model(
        inputs=text_input,
        outputs={
            "merchant": merchant_output,
            "category": category_output
        },
        name="SpendSenseClassifier"
    )

    # --------------------------------------------------------
    # Compile
    # --------------------------------------------------------

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss={
            "merchant":
                tf.keras.losses.SparseCategoricalCrossentropy(),

            "category":
                tf.keras.losses.SparseCategoricalCrossentropy(),
        },
        loss_weights={
            "merchant": 0.6,
            "category": 1.0,
        },
        metrics={
            "merchant": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],

            "category": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ]
        }
    )

    print("\nModel Summary:\n")

    model.summary()

    return model


# ============================================================
# CREATE TF.DATA DATASET
# ============================================================

def create_tf_dataset(
    texts,
    targets,
    training=False
):

    texts = (
        np.asarray(texts)
        .astype(str)
        .reshape(-1, 1)
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            texts,
            {
                "merchant": targets["merchant"],
                "category": targets["category"],
            }
        )
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=min(
                len(texts),
                10000
            ),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


# ============================================================
# TRAIN MODEL
# ============================================================

def train_model(
    model,
    train_dataset,
    val_dataset,
    output_dir
):

    print("\n" + "=" * 70)
    print("TRAINING MODEL")
    print("=" * 70)

    checkpoint_path = (
        Path(output_dir) /
        "best_model.keras"
    )

    callbacks = [

        tf.keras.callbacks.EarlyStopping(
            monitor="val_category_accuracy",
            mode="max",
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_category_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # Save history
    history_data = {}

    for key, values in history.history.items():

        history_data[key] = [
            float(v)
            for v in values
        ]

    history_path = (
        Path(output_dir) /
        "training_history.json"
    )

    with open(history_path, "w") as f:

        json.dump(
            history_data,
            f,
            indent=2
        )

    print(
        f"\n✓ Training history saved: "
        f"{history_path}"
    )

    return history


# ============================================================
# EVALUATE MODEL
# ============================================================

def evaluate_model(
    model,
    test_df,
    encoders,
    output_dir
):

    print("\n" + "=" * 70)
    print("FINAL EVALUATION")
    print("=" * 70)

    test_texts = (
        test_df["transaction_text"]
        .astype(str)
        .values
        .reshape(-1, 1)
    )

    predictions = model.predict(
        test_texts,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    test_targets = prepare_targets(
        test_df,
        encoders
    )

    results = {}

    for task in ["merchant", "category"]:

        print("\n" + "-" * 70)
        print(
            f"{task.upper()} EVALUATION"
        )
        print("-" * 70)

        probabilities = predictions[task]

        y_pred = np.argmax(
            probabilities,
            axis=1
        )

        y_true = test_targets[task]

        accuracy = accuracy_score(
            y_true,
            y_pred
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )

        print(
            f"\nAccuracy:    {accuracy:.4f}"
        )

        print(
            f"Macro F1:    {macro_f1:.4f}"
        )

        print(
            f"Weighted F1: {weighted_f1:.4f}"
        )

        results[task] = {
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
        }

        # Classification report

        report = classification_report(
            y_true,
            y_pred,
            target_names=encoders[task].classes_,
            zero_division=0
        )

        report_path = (
            Path(output_dir) /
            f"{task}_classification_report.txt"
        )

        with open(
            report_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(report)

        # Confusion matrix

        cm = confusion_matrix(
            y_true,
            y_pred
        )

        cm_path = (
            Path(output_dir) /
            f"{task}_confusion_matrix.csv"
        )

        pd.DataFrame(cm).to_csv(
            cm_path,
            index=False
        )

        print(
            f"✓ Report saved: {report_path}"
        )

    results_path = (
        Path(output_dir) /
        "evaluation_results.json"
    )

    with open(results_path, "w") as f:

        json.dump(
            results,
            f,
            indent=2
        )

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    for task, metrics in results.items():

        print(f"\n{task.upper()}")

        for metric, value in metrics.items():

            print(
                f"{metric}: {value:.4f}"
            )

    return results


# ============================================================
# EXPORT TFLITE
# ============================================================

def export_tflite(model, output_dir):

    print("\n" + "=" * 70)
    print("EXPORTING TFLITE MODELS")
    print("=" * 70)

    output_dir = Path(output_dir)

    # --------------------------------------------------------
    # Float32
    # --------------------------------------------------------

    print("\nExporting Float32 model...")

    converter = tf.lite.TFLiteConverter.from_keras_model(
        model
    )

    tflite_model = converter.convert()

    float32_path = (
        output_dir /
        "spendsense_float32.tflite"
    )

    with open(
        float32_path,
        "wb"
    ) as f:

        f.write(tflite_model)

    float32_size = (
        float32_path.stat().st_size /
        (1024 * 1024)
    )

    print(
        f"✓ Float32 saved: {float32_path}"
    )

    print(
        f"  Size: {float32_size:.2f} MB"
    )

    # --------------------------------------------------------
    # Float16
    # --------------------------------------------------------

    print("\nExporting Float16 model...")

    converter = tf.lite.TFLiteConverter.from_keras_model(
        model
    )

    converter.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]

    converter.target_spec.supported_types = [
        tf.float16
    ]

    tflite_model_fp16 = converter.convert()

    float16_path = (
        output_dir /
        "spendsense_float16.tflite"
    )

    with open(
        float16_path,
        "wb"
    ) as f:

        f.write(tflite_model_fp16)

    float16_size = (
        float16_path.stat().st_size /
        (1024 * 1024)
    )

    print(
        f"✓ Float16 saved: {float16_path}"
    )

    print(
        f"  Size: {float16_size:.2f} MB"
    )


# ============================================================
# VERIFY TFLITE
# ============================================================

def verify_tflite(output_dir):

    print("\n" + "=" * 70)
    print("VERIFYING TFLITE MODEL")
    print("=" * 70)

    tflite_path = (
        Path(output_dir) /
        "spendsense_float16.tflite"
    )

    interpreter = tf.lite.Interpreter(
        model_path=str(tflite_path)
    )

    interpreter.allocate_tensors()

    print("\nInputs:")

    for detail in interpreter.get_input_details():

        print(
            f"\nName: {detail['name']}"
        )

        print(
            f"Shape: {detail['shape']}"
        )

        print(
            f"Type: {detail['dtype']}"
        )

    print("\nOutputs:")

    for detail in interpreter.get_output_details():

        print(
            f"\nName: {detail['name']}"
        )

        print(
            f"Shape: {detail['shape']}"
        )

        print(
            f"Type: {detail['dtype']}"
        )

    print(
        "\n✓ TFLite verification successful"
    )


# ============================================================
# SAVE CONFIG
# ============================================================

def save_config(output_dir, encoders):

    config = {

        "model_name":
            "SpendSenseTransactionClassifier",

        "input_column":
            "transaction_text",

        "input":
            "Parsed transaction narration",

        "outputs": {
            "merchant":
                len(encoders["merchant"].classes_),

            "category":
                len(encoders["category"].classes_),
        },

        "architecture": {
            "tokenization":
                "character-level",

            "max_vocab_size":
                MAX_VOCAB_SIZE,

            "max_sequence_length":
                MAX_SEQUENCE_LENGTH,

            "embedding_dim":
                EMBEDDING_DIM,

            "cnn_filters":
                CNN_FILTERS,

            "kernel_sizes":
                [3, 5, 7],

            "dense_units":
                DENSE_UNITS,

            "dropout":
                DROPOUT_RATE,
        },

        "training": {
            "batch_size":
                BATCH_SIZE,

            "epochs":
                EPOCHS,

            "learning_rate":
                LEARNING_RATE,

            "seed":
                SEED,
        }
    }

    config_path = (
        Path(output_dir) /
        "model_config.json"
    )

    with open(
        config_path,
        "w"
    ) as f:

        json.dump(
            config,
            f,
            indent=2
        )

    print(
        f"\n✓ Config saved: {config_path}"
    )


# ============================================================
# MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # Setup
    # --------------------------------------------------------

    set_seed()

    Path(MODEL_DIR).mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 70)
    print("SPENDSENSE MODEL TRAINING")
    print("=" * 70)

    print(
        f"\nTensorFlow Version: "
        f"{tf.__version__}"
    )

    print(
        f"Dataset Path: "
        f"{DATA_PATH}"
    )

    print(
        f"Output Directory: "
        f"{MODEL_DIR}"
    )

    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------

    df = load_dataset()

    # --------------------------------------------------------
    # Validate dataset
    # --------------------------------------------------------

    validate_dataset(df)

    # --------------------------------------------------------
    # Create label encoders
    # --------------------------------------------------------

    encoders = create_label_encoders(
        df,
        MODEL_DIR
    )

    # --------------------------------------------------------
    # Split dataset
    # --------------------------------------------------------

    train_df, val_df, test_df = split_dataset(
        df
    )

    # --------------------------------------------------------
    # Create vectorizer
    # --------------------------------------------------------

    vectorizer = create_vectorizer(
        train_df["transaction_text"]
        .astype(str)
        .values,
        MODEL_DIR
    )

    # --------------------------------------------------------
    # Prepare targets
    # --------------------------------------------------------

    train_targets = prepare_targets(
        train_df,
        encoders
    )

    val_targets = prepare_targets(
        val_df,
        encoders
    )

    # --------------------------------------------------------
    # Create TensorFlow datasets
    # --------------------------------------------------------

    train_dataset = create_tf_dataset(
        train_df["transaction_text"]
        .values,
        train_targets,
        training=True
    )

    val_dataset = create_tf_dataset(
        val_df["transaction_text"]
        .values,
        val_targets,
        training=False
    )

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    model = build_model(
        vectorizer,
        encoders
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train_model(
        model,
        train_dataset,
        val_dataset,
        MODEL_DIR
    )

    # --------------------------------------------------------
    # Save Keras model
    # --------------------------------------------------------

    keras_path = (
        Path(MODEL_DIR) /
        "spendsense_model.keras"
    )

    model.save(
        keras_path
    )

    print(
        f"\n✓ Keras model saved: "
        f"{keras_path}"
    )

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    evaluate_model(
        model,
        test_df,
        encoders,
        MODEL_DIR
    )

    # --------------------------------------------------------
    # Save config
    # --------------------------------------------------------

    save_config(
        MODEL_DIR,
        encoders
    )

    # --------------------------------------------------------
    # Export TFLite
    # --------------------------------------------------------

    export_tflite(
        model,
        MODEL_DIR
    )

    # --------------------------------------------------------
    # Verify TFLite
    # --------------------------------------------------------

    verify_tflite(
        MODEL_DIR
    )

    # --------------------------------------------------------
    # Complete
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)

    print("\nGenerated files:\n")

    for file in sorted(
        Path(MODEL_DIR).rglob("*")
    ):

        if file.is_file():

            size_kb = (
                file.stat().st_size / 1024
            )

            print(
                f"✓ {file.name} "
                f"({size_kb:.2f} KB)"
            )


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()

Overwriting train_spendsense.py


In [ ]:
!python train_spendsense.py

✓ Random seed set to 42

SPENDSENSE MODEL TRAINING

TensorFlow Version: 2.20.0
Dataset Path: /content/indian_payment_transaction_dataset (1).csv
Output Directory: /content/spendsense_model_output

LOADING DATASET

Dataset Shape: (133131, 5)

Dataset Columns:
['transaction_text', 'merchant', 'category', 'category_id', 'augmentation_type']

✓ Required columns validated
Removed empty transaction rows: 0
Removed exact duplicates: 25481

Final Dataset Size: 107,650

DATASET VALIDATION

Merchant Statistics:
Unique merchants: 734
Minimum samples per merchant: 1
Maximum samples per merchant: 511

Category Statistics:
Unique categories: 15

Category Distribution:
category
Personal         10531
Shopping         10455
Food             10068
Utilities         7319
Subscription      7213
Education         7080
Medical           6946
Groceries         6756
Travel            6710
Transport         6622
Housing           6297
Entertainment     5869
EMI               5296
Income            5255
Fuel  

In [ ]:
%%writefile evaluate_spendsense.py

"""
============================================================
SpendSense Model Evaluation Pipeline
============================================================

This script DOES NOT train the model.

It:
1. Loads the already trained SpendSense Keras model
2. Recreates the exact test split
3. Loads label mappings
4. Runs predictions
5. Calculates evaluation metrics
6. Generates classification reports
7. Generates confusion matrices
8. Saves correct/incorrect predictions
9. Generates visualization plots
10. Creates a final evaluation summary

Output:
/content/spendsense_evaluation/
============================================================
"""

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

# Dataset
DATA_PATH = "/content/indian_payment_transaction_dataset (1).csv"

# Trained model
MODEL_PATH = (
    "/content/spendsense_model_output/"
    "spendsense_model.keras"
)

# Label directory
LABEL_DIR = (
    "/content/spendsense_model_output/"
    "labels"
)

# Evaluation output
EVALUATION_DIR = (
    "/content/spendsense_evaluation"
)

# Same split configuration used during training
VALIDATION_SIZE = 0.10
TEST_SIZE = 0.10

BATCH_SIZE = 128

# Number of classes shown in confusion matrix plot
TOP_CLASSES_CONFUSION_MATRIX = 30


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    print(f"✓ Random seed set to {seed}")


# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

def create_directories():

    base = Path(EVALUATION_DIR)

    directories = {
        "base": base,

        "merchant": base / "merchant",

        "category": base / "category",

        "predictions": base / "predictions",

        "plots": base / "plots",
    }

    for directory in directories.values():

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    print("\n✓ Evaluation directories created")

    return directories


# ============================================================
# LOAD DATASET
# ============================================================

def load_dataset():

    print("\n" + "=" * 70)
    print("LOADING DATASET")
    print("=" * 70)

    if not os.path.exists(DATA_PATH):

        print(f"\nDataset not found: {DATA_PATH}")

        print("\nAvailable files:")

        for file in os.listdir("/content"):
            print(" -", file)

        raise FileNotFoundError(DATA_PATH)

    df = pd.read_csv(DATA_PATH)

    required_columns = [
        "transaction_text",
        "merchant",
        "category"
    ]

    missing_columns = [

        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    df = df[required_columns].copy()

    # Clean text exactly like training
    df["transaction_text"] = (
        df["transaction_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Clean merchant
    df["merchant"] = (
        df["merchant"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

    # Clean category
    df["category"] = (
        df["category"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

    # Remove empty transaction text
    df = df[
        df["transaction_text"].str.len() > 0
    ].copy()

    # Remove duplicates exactly like training
    df = df.drop_duplicates()

    # Shuffle exactly like training
    df = df.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    print(f"\nFinal Dataset Size: {len(df):,}")

    print("\nColumns:")
    print(df.columns.tolist())

    return df


# ============================================================
# RECREATE TEST SPLIT
# ============================================================

def recreate_test_split(df):

    print("\n" + "=" * 70)
    print("RECREATING ORIGINAL TEST SPLIT")
    print("=" * 70)

    # Must match training script exactly

    train_df, temp_df = train_test_split(

        df,

        test_size=(
            VALIDATION_SIZE +
            TEST_SIZE
        ),

        random_state=SEED,

        stratify=df["category"]
    )

    val_df, test_df = train_test_split(

        temp_df,

        test_size=0.5,

        random_state=SEED,

        stratify=temp_df["category"]
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"\nTrain samples: {len(train_df):,}")
    print(f"Validation samples: {len(val_df):,}")
    print(f"Test samples: {len(test_df):,}")

    return test_df


# ============================================================
# LOAD LABELS
# ============================================================

def load_labels():

    print("\n" + "=" * 70)
    print("LOADING LABEL MAPPINGS")
    print("=" * 70)

    labels = {}

    for task in ["merchant", "category"]:

        label_path = (
            Path(LABEL_DIR) /
            f"{task}_labels.json"
        )

        if not label_path.exists():

            raise FileNotFoundError(
                f"Label file missing: {label_path}"
            )

        with open(
            label_path,
            "r",
            encoding="utf-8"
        ) as f:

            labels[task] = json.load(f)

        print(
            f"✓ {task}: "
            f"{len(labels[task])} classes"
        )

    return labels


# ============================================================
# LOAD MODEL
# ============================================================

def load_model():

    print("\n" + "=" * 70)
    print("LOADING TRAINED MODEL")
    print("=" * 70)

    if not os.path.exists(MODEL_PATH):

        raise FileNotFoundError(
            f"Model not found: {MODEL_PATH}"
        )

    model = tf.keras.models.load_model(
        MODEL_PATH,
        compile=False
    )

    print("✓ Model loaded successfully")

    print("\nModel Inputs:")

    for inp in model.inputs:

        print(
            f"  {inp.name} "
            f"shape={inp.shape} "
            f"dtype={inp.dtype}"
        )

    print("\nModel Outputs:")

    for output in model.outputs:

        print(
            f"  {output.name} "
            f"shape={output.shape}"
        )

    return model


# ============================================================
# SAFE PREDICTION
# ============================================================

def run_predictions(model, test_df):

    print("\n" + "=" * 70)
    print("RUNNING MODEL PREDICTIONS")
    print("=" * 70)

    # IMPORTANT:
    # Convert to TensorFlow string tensor.
    # This avoids:
    # ValueError: Invalid dtype: object

    test_texts = (
        test_df["transaction_text"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    test_tensor = tf.constant(
        test_texts,
        dtype=tf.string
    )

    test_tensor = tf.expand_dims(
        test_tensor,
        axis=1
    )

    print(
        f"\nInput Tensor Shape: "
        f"{test_tensor.shape}"
    )

    print(
        f"Input Tensor dtype: "
        f"{test_tensor.dtype}"
    )

    # Direct model call avoids Keras
    # NumPy object dtype issues

    predictions = model(
        test_tensor,
        training=False
    )

    # Convert tensors to numpy

    if isinstance(predictions, dict):

        predictions = {

            key: value.numpy()

            for key, value
            in predictions.items()
        }

    elif isinstance(predictions, (list, tuple)):

        predictions = {

            "merchant": predictions[0].numpy(),

            "category": predictions[1].numpy()
        }

    else:

        raise ValueError(
            "Unexpected model output format"
        )

    print("\n✓ Predictions completed")

    for key, value in predictions.items():

        print(
            f"{key}: {value.shape}"
        )

    return predictions


# ============================================================
# CONVERT TRUE LABELS TO IDS
# ============================================================

def encode_labels(test_df, labels):

    encoded = {}

    for task in ["merchant", "category"]:

        label_to_id = {

            label: index

            for index, label
            in enumerate(labels[task])
        }

        encoded[task] = np.array(

            [
                label_to_id[value]

                for value
                in test_df[task].astype(str)
            ],

            dtype=np.int32
        )

    return encoded


# ============================================================
# CALCULATE METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    metrics = {

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),

        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        ),

        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        ),

        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )
    }

    return metrics


# ============================================================
# SAVE CLASSIFICATION REPORT
# ============================================================

def save_classification_reports(
    y_true,
    y_pred,
    labels,
    task,
    directories
):

    task_dir = directories[task]

    # Text report

    report_text = classification_report(

        y_true,

        y_pred,

        labels=np.arange(len(labels)),

        target_names=labels,

        zero_division=0
    )

    text_path = (
        task_dir /
        "classification_report.txt"
    )

    with open(
        text_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(report_text)

    # CSV report

    report_dict = classification_report(

        y_true,

        y_pred,

        labels=np.arange(len(labels)),

        target_names=labels,

        output_dict=True,

        zero_division=0
    )

    report_df = (
        pd.DataFrame(report_dict)
        .transpose()
    )

    csv_path = (
        task_dir /
        "classification_report.csv"
    )

    report_df.to_csv(csv_path)

    # Per class metrics only

    class_metrics = report_df.iloc[
        :len(labels)
    ].copy()

    per_class_path = (
        task_dir /
        "per_class_metrics.csv"
    )

    class_metrics.to_csv(per_class_path)

    print(
        f"✓ {task} classification reports saved"
    )

    return report_df


# ============================================================
# SAVE CONFUSION MATRIX
# ============================================================

def save_confusion_matrix(
    y_true,
    y_pred,
    labels,
    task,
    directories
):

    task_dir = directories[task]

    # Full confusion matrix

    cm = confusion_matrix(

        y_true,

        y_pred,

        labels=np.arange(len(labels))
    )

    cm_df = pd.DataFrame(

        cm,

        index=labels,

        columns=labels
    )

    csv_path = (
        task_dir /
        "confusion_matrix.csv"
    )

    cm_df.to_csv(csv_path)

    print(
        f"✓ {task} full confusion matrix saved"
    )

    # --------------------------------------------------------
    # Plot top classes only
    # --------------------------------------------------------

    class_counts = np.bincount(

        y_true,

        minlength=len(labels)
    )

    top_indices = np.argsort(
        class_counts
    )[::-1][
        :TOP_CLASSES_CONFUSION_MATRIX
    ]

    y_true_filtered = []
    y_pred_filtered = []

    top_set = set(top_indices)

    for true, pred in zip(
        y_true,
        y_pred
    ):

        if true in top_set:

            y_true_filtered.append(true)

            if pred in top_set:
                y_pred_filtered.append(pred)
            else:
                y_pred_filtered.append(true)

    filtered_cm = confusion_matrix(

        y_true_filtered,

        y_pred_filtered,

        labels=top_indices
    )

    plt.figure(
        figsize=(14, 12)
    )

    display = ConfusionMatrixDisplay(

        confusion_matrix=filtered_cm,

        display_labels=[
            labels[i]
            for i in top_indices
        ]
    )

    display.plot(
        xticks_rotation=90,
        values_format="d"
    )

    plt.title(
        f"SpendSense {task.title()} "
        f"Confusion Matrix (Top Classes)"
    )

    plt.tight_layout()

    plot_path = (
        task_dir /
        "confusion_matrix.png"
    )

    plt.savefig(
        plot_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"✓ {task} confusion matrix plot saved"
    )


# ============================================================
# SAVE ERROR ANALYSIS
# ============================================================

def save_error_analysis(
    test_df,
    predictions,
    true_labels,
    predicted_labels,
    task,
    directories
):

    task_dir = directories[task]

    analysis_df = pd.DataFrame({

        "transaction_text":
            test_df[
                "transaction_text"
            ].values,

        "true_label":
            true_labels,

        "predicted_label":
            predicted_labels,

        "correct":
            (
                np.array(true_labels)
                ==
                np.array(predicted_labels)
            )
    })

    # Incorrect predictions

    errors = analysis_df[
        ~analysis_df["correct"]
    ].copy()

    errors_path = (
        task_dir /
        "top_errors.csv"
    )

    errors.to_csv(
        errors_path,
        index=False
    )

    print(
        f"✓ {task} error analysis saved"
    )

    return errors


# ============================================================
# CREATE ALL PREDICTIONS FILE
# ============================================================

def save_all_predictions(
    test_df,
    prediction_data,
    directories
):

    output = test_df.copy()

    for task in ["merchant", "category"]:

        output[
            f"predicted_{task}"
        ] = prediction_data[
            task
        ]["predicted_labels"]

        output[
            f"{task}_confidence"
        ] = prediction_data[
            task
        ]["confidence"]

        output[
            f"{task}_correct"
        ] = (

            output[task].astype(str)
            ==
            output[
                f"predicted_{task}"
            ].astype(str)
        )

    predictions_dir = (
        directories["predictions"]
    )

    all_path = (
        predictions_dir /
        "all_test_predictions.csv"
    )

    output.to_csv(
        all_path,
        index=False
    )

    # Any incorrect prediction

    incorrect = output[
        (~output["merchant_correct"])
        |
        (~output["category_correct"])
    ].copy()

    incorrect_path = (
        predictions_dir /
        "incorrect_predictions.csv"
    )

    incorrect.to_csv(
        incorrect_path,
        index=False
    )

    print(
        "\n✓ All predictions saved"
    )

    print(
        f"✓ Incorrect predictions: "
        f"{len(incorrect):,}"
    )


# ============================================================
# DATA DISTRIBUTION PLOTS
# ============================================================

def save_distribution_plots(
    test_df,
    directories
):

    plots_dir = directories["plots"]

    for column in [
        "category",
        "merchant"
    ]:

        counts = (
            test_df[column]
            .value_counts()
        )

        # Merchant plots can become unreadable
        # Limit visualization

        if column == "merchant":

            counts = counts.head(30)

        plt.figure(
            figsize=(14, 7)
        )

        plt.bar(
            range(len(counts)),
            counts.values
        )

        plt.xticks(

            range(len(counts)),

            counts.index,

            rotation=90
        )

        plt.xlabel(
            column.title()
        )

        plt.ylabel(
            "Number of Samples"
        )

        plt.title(
            f"Test Dataset {column.title()} "
            f"Distribution"
        )

        plt.tight_layout()

        plot_path = (
            plots_dir /
            f"{column}_distribution.png"
        )

        plt.savefig(
            plot_path,
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

        print(
            f"✓ {column} distribution plot saved"
        )


# ============================================================
# TRAINING HISTORY PLOT
# ============================================================

def save_training_history_plot(
    directories
):

    history_path = (
        Path(
            "/content/spendsense_model_output"
        )
        /
        "training_history.json"
    )

    if not history_path.exists():

        print(
            "\n⚠ Training history not found."
        )

        return

    with open(
        history_path,
        "r"
    ) as f:

        history = json.load(f)

    plots_dir = directories["plots"]

    # Plot loss

    if (
        "loss" in history
        and "val_loss" in history
    ):

        plt.figure(
            figsize=(10, 6)
        )

        plt.plot(
            history["loss"],
            label="Training Loss"
        )

        plt.plot(
            history["val_loss"],
            label="Validation Loss"
        )

        plt.xlabel("Epoch")

        plt.ylabel("Loss")

        plt.title(
            "SpendSense Training vs Validation Loss"
        )

        plt.legend()

        plt.grid()

        plt.tight_layout()

        plt.savefig(
            plots_dir /
            "training_loss.png",
            dpi=200
        )

        plt.close()

    # Plot category accuracy

    category_accuracy = None
    val_category_accuracy = None

    for key in history:

        if (
            "category_accuracy" in key
            and not key.startswith("val")
        ):

            category_accuracy = history[key]

        if (
            "val_category_accuracy" in key
        ):

            val_category_accuracy = history[key]

    if (
        category_accuracy is not None
        and val_category_accuracy is not None
    ):

        plt.figure(
            figsize=(10, 6)
        )

        plt.plot(
            category_accuracy,
            label="Training Accuracy"
        )

        plt.plot(
            val_category_accuracy,
            label="Validation Accuracy"
        )

        plt.xlabel("Epoch")

        plt.ylabel("Accuracy")

        plt.title(
            "SpendSense Category Accuracy"
        )

        plt.legend()

        plt.grid()

        plt.tight_layout()

        plt.savefig(
            plots_dir /
            "category_accuracy.png",
            dpi=200
        )

        plt.close()

    print(
        "✓ Training history plots saved"
    )


# ============================================================
# CREATE FINAL SUMMARY
# ============================================================

def create_summary(
    metrics,
    test_size,
    labels,
    directories
):

    print("\n" + "=" * 70)
    print("CREATING FINAL EVALUATION SUMMARY")
    print("=" * 70)

    summary = []

    summary.append(
        "=" * 70
    )

    summary.append(
        "SPENDSENSE MODEL EVALUATION REPORT"
    )

    summary.append(
        "=" * 70
    )

    summary.append("")

    summary.append(
        f"Test Samples: {test_size:,}"
    )

    summary.append(
        f"Merchant Classes: "
        f"{len(labels['merchant'])}"
    )

    summary.append(
        f"Category Classes: "
        f"{len(labels['category'])}"
    )

    summary.append("")

    summary.append(
        "=" * 70
    )

    summary.append(
        "MERCHANT CLASSIFICATION"
    )

    summary.append(
        "=" * 70
    )

    for key, value in metrics[
        "merchant"
    ].items():

        summary.append(
            f"{key}: {value:.4f}"
        )

    summary.append("")

    summary.append(
        "=" * 70
    )

    summary.append(
        "CATEGORY CLASSIFICATION"
    )

    summary.append(
        "=" * 70
    )

    for key, value in metrics[
        "category"
    ].items():

        summary.append(
            f"{key}: {value:.4f}"
        )

    summary.append("")

    summary.append(
        "=" * 70
    )

    summary.append(
        "INTERPRETATION"
    )

    summary.append(
        "=" * 70
    )

    category_accuracy = metrics[
        "category"
    ]["accuracy"]

    if category_accuracy >= 0.97:

        interpretation = (
            "Excellent category classification "
            "performance."
        )

    elif category_accuracy >= 0.94:

        interpretation = (
            "Strong category classification "
            "performance. Meets the target range."
        )

    elif category_accuracy >= 0.90:

        interpretation = (
            "Good performance, but additional "
            "dataset improvements may increase "
            "generalization."
        )

    else:

        interpretation = (
            "Performance requires further "
            "dataset and architecture analysis."
        )

    summary.append(
        interpretation
    )

    summary_text = "\n".join(
        summary
    )

    summary_path = (
        directories["base"] /
        "evaluation_summary.txt"
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(summary_text)

    print(summary_text)

    # Save JSON metrics

    json_path = (
        directories["base"] /
        "evaluation_results.json"
    )

    with open(
        json_path,
        "w"
    ) as f:

        json.dump(
            metrics,
            f,
            indent=4
        )

    print(
        f"\n✓ Summary saved: "
        f"{summary_path}"
    )


# ============================================================
# MAIN EVALUATION PIPELINE
# ============================================================

def main():

    print("\n" + "=" * 70)
    print("SPENDSENSE SEPARATE EVALUATION PIPELINE")
    print("=" * 70)

    set_seed()

    # --------------------------------------------------------
    # Create folders
    # --------------------------------------------------------

    directories = create_directories()

    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------

    df = load_dataset()

    # --------------------------------------------------------
    # Recreate original test split
    # --------------------------------------------------------

    test_df = recreate_test_split(
        df
    )

    # --------------------------------------------------------
    # Load labels
    # --------------------------------------------------------

    labels = load_labels()

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    model = load_model()

    # --------------------------------------------------------
    # Run predictions
    # --------------------------------------------------------

    predictions = run_predictions(
        model,
        test_df
    )

    # --------------------------------------------------------
    # Encode true labels
    # --------------------------------------------------------

    encoded_true = encode_labels(
        test_df,
        labels
    )

    # --------------------------------------------------------
    # Store metrics
    # --------------------------------------------------------

    all_metrics = {}

    prediction_data = {}

    # --------------------------------------------------------
    # Evaluate both outputs
    # --------------------------------------------------------

    for task in ["merchant", "category"]:

        print("\n" + "=" * 70)
        print(
            f"EVALUATING {task.upper()}"
        )
        print("=" * 70)

        probabilities = predictions[task]

        y_pred = np.argmax(
            probabilities,
            axis=1
        )

        confidence = np.max(
            probabilities,
            axis=1
        )

        y_true = encoded_true[task]

        # Metrics

        metrics = calculate_metrics(
            y_true,
            y_pred
        )

        all_metrics[task] = metrics

        print("\nMetrics:")

        for key, value in metrics.items():

            print(
                f"{key}: {value:.4f}"
            )

        # Convert IDs to labels

        true_labels = [

            labels[task][index]

            for index in y_true
        ]

        predicted_labels = [

            labels[task][index]

            for index in y_pred
        ]

        # Save reports

        save_classification_reports(

            y_true,

            y_pred,

            labels[task],

            task,

            directories
        )

        # Save confusion matrix

        save_confusion_matrix(

            y_true,

            y_pred,

            labels[task],

            task,

            directories
        )

        # Save error analysis

        save_error_analysis(

            test_df,

            probabilities,

            true_labels,

            predicted_labels,

            task,

            directories
        )

        # Store prediction data

        prediction_data[task] = {

            "predicted_labels":
                predicted_labels,

            "confidence":
                confidence
        }

    # --------------------------------------------------------
    # Save all predictions
    # --------------------------------------------------------

    save_all_predictions(

        test_df,

        prediction_data,

        directories
    )

    # --------------------------------------------------------
    # Dataset distribution
    # --------------------------------------------------------

    save_distribution_plots(

        test_df,

        directories
    )

    # --------------------------------------------------------
    # Training history
    # --------------------------------------------------------

    save_training_history_plot(
        directories
    )

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    create_summary(

        all_metrics,

        len(test_df),

        labels,

        directories
    )

    # --------------------------------------------------------
    # Complete
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("EVALUATION COMPLETE")
    print("=" * 70)

    print(
        f"\nAll evaluation files saved to:\n"
        f"{EVALUATION_DIR}"
    )

    print("\nFolder Structure:")

    for path in sorted(
        Path(EVALUATION_DIR).rglob("*")
    ):

        if path.is_file():

            print(
                f"✓ {path.relative_to(EVALUATION_DIR)}"
            )


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()

Overwriting evaluate_spendsense.py


In [ ]:
!python evaluate_spendsense.py


SPENDSENSE SEPARATE EVALUATION PIPELINE
✓ Random seed set to 42

✓ Evaluation directories created

LOADING DATASET

Final Dataset Size: 107,650

Columns:
['transaction_text', 'merchant', 'category']

RECREATING ORIGINAL TEST SPLIT

Train samples: 86,120
Validation samples: 10,765
Test samples: 10,765

LOADING LABEL MAPPINGS
✓ merchant: 734 classes
✓ category: 15 classes

LOADING TRAINED MODEL
2026-09-02 19:22:02.747769: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788376922.749266   17778 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
✓ Model loaded successfully

Model Inputs:
  transaction_text shape=(None, 1) dtype=string

Model Outputs:
  keras_tensor_25 shape=(None, 15)
  keras_tensor_27 shape=

In [ ]:
%%writefile convert_spendsense_tflite.py

"""
============================================================
SpendSense TFLite Conversion & Validation Pipeline
============================================================

Purpose:
1. Read evaluation results
2. Load trained Keras model
3. Export Float32 TFLite
4. Export Dynamic Range Quantized TFLite
5. Validate TFLite models
6. Compare Keras vs TFLite predictions
7. Generate deployment recommendation

Input:
    /content/spendsense_model_output/
    /content/spendsense_evaluation/

Output:
    /content/spendsense_tflite/
============================================================
"""

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_DIR = Path(
    "/content/spendsense_model_output"
)

EVALUATION_DIR = Path(
    "/content/spendsense_evaluation"
)

OUTPUT_DIR = Path(
    "/content/spendsense_tflite"
)

MODEL_PATH = MODEL_DIR / "spendsense_model.keras"

LABEL_DIR = MODEL_DIR / "labels"

EVALUATION_RESULTS_PATH = (
    EVALUATION_DIR /
    "evaluation_results.json"
)

PREDICTIONS_PATH = (
    EVALUATION_DIR /
    "predictions" /
    "all_test_predictions.csv"
)

BATCH_SIZE = 128

# Number of samples used for Keras vs TFLite comparison
VALIDATION_SAMPLES = 500


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

def setup_directories():

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        f"✓ Output directory ready: {OUTPUT_DIR}"
    )


# ============================================================
# READ EVALUATION RESULTS
# ============================================================

def inspect_evaluation():

    print("\n" + "=" * 70)
    print("INSPECTING MODEL EVALUATION RESULTS")
    print("=" * 70)

    evaluation_data = {}

    # --------------------------------------------------------
    # Read overall metrics
    # --------------------------------------------------------

    if EVALUATION_RESULTS_PATH.exists():

        with open(
            EVALUATION_RESULTS_PATH,
            "r"
        ) as f:

            evaluation_data = json.load(f)

        print("\nEvaluation Results:\n")

        for task, metrics in evaluation_data.items():

            print(
                f"{task.upper()}"
            )

            for metric, value in metrics.items():

                print(
                    f"  {metric}: {value:.4f}"
                )

    else:

        print(
            "\n⚠ evaluation_results.json not found"
        )

    # --------------------------------------------------------
    # Check prediction file
    # --------------------------------------------------------

    prediction_df = None

    if PREDICTIONS_PATH.exists():

        prediction_df = pd.read_csv(
            PREDICTIONS_PATH
        )

        print(
            f"\n✓ Evaluation predictions loaded"
        )

        print(
            f"Test samples available: "
            f"{len(prediction_df):,}"
        )

        # Calculate error counts

        for task in ["merchant", "category"]:

            correct_column = (
                f"{task}_correct"
            )

            if correct_column in prediction_df.columns:

                accuracy = (
                    prediction_df[
                        correct_column
                    ]
                    .mean()
                )

                print(
                    f"{task} prediction accuracy "
                    f"from evaluation file: "
                    f"{accuracy:.4f}"
                )

    else:

        print(
            "\n⚠ Prediction CSV not found."
        )

    return (
        evaluation_data,
        prediction_df
    )


# ============================================================
# LOAD LABELS
# ============================================================

def load_labels():

    labels = {}

    print("\n" + "=" * 70)
    print("LOADING LABELS")
    print("=" * 70)

    for task in [
        "merchant",
        "category"
    ]:

        label_path = (
            LABEL_DIR /
            f"{task}_labels.json"
        )

        if not label_path.exists():

            raise FileNotFoundError(
                f"Missing labels: {label_path}"
            )

        with open(
            label_path,
            "r",
            encoding="utf-8"
        ) as f:

            labels[task] = json.load(f)

        print(
            f"✓ {task}: "
            f"{len(labels[task])} classes"
        )

    return labels


# ============================================================
# LOAD KERAS MODEL
# ============================================================

def load_keras_model():

    print("\n" + "=" * 70)
    print("LOADING TRAINED KERAS MODEL")
    print("=" * 70)

    if not MODEL_PATH.exists():

        raise FileNotFoundError(
            f"Model not found: {MODEL_PATH}"
        )

    model = tf.keras.models.load_model(
        MODEL_PATH,
        compile=False
    )

    print("✓ Keras model loaded")

    print("\nModel Input:")

    for inp in model.inputs:

        print(
            f"  Name: {inp.name}"
        )

        print(
            f"  Shape: {inp.shape}"
        )

        print(
            f"  Dtype: {inp.dtype}"
        )

    print("\nModel Outputs:")

    for output in model.outputs:

        print(
            f"  Name: {output.name}"
        )

        print(
            f"  Shape: {output.shape}"
        )

    return model


# ============================================================
# SAVE MODEL INFORMATION
# ============================================================

def inspect_model(model):

    print("\n" + "=" * 70)
    print("MODEL ARCHITECTURE INSPECTION")
    print("=" * 70)

    layers_info = []

    for layer in model.layers:

        layer_data = {

            "name": layer.name,

            "type":
                layer.__class__.__name__,

            "trainable_params":
                int(
                    layer.count_params()
                )
                if hasattr(
                    layer,
                    "count_params"
                )
                else 0
        }

        layers_info.append(
            layer_data
        )

        print(
            f"{layer.name:<30} "
            f"{layer.__class__.__name__}"
        )

    model_info = {

        "total_parameters":
            int(model.count_params()),

        "layers":
            layers_info
    }

    output_path = (
        OUTPUT_DIR /
        "model_architecture.json"
    )

    with open(
        output_path,
        "w"
    ) as f:

        json.dump(
            model_info,
            f,
            indent=4
        )

    print(
        f"\nTotal parameters: "
        f"{model.count_params():,}"
    )

    return model_info


# ============================================================
# EXPORT FLOAT32 TFLITE
# ============================================================

def export_float32(model):

    print("\n" + "=" * 70)
    print("EXPORTING FLOAT32 TFLITE")
    print("=" * 70)

    output_path = (
        OUTPUT_DIR /
        "spendsense_float32.tflite"
    )

    try:

        converter = (
            tf.lite.TFLiteConverter
            .from_keras_model(model)
        )

        tflite_model = converter.convert()

        with open(
            output_path,
            "wb"
        ) as f:

            f.write(tflite_model)

        size_mb = (
            output_path.stat().st_size /
            (1024 * 1024)
        )

        print(
            f"✓ Float32 TFLite created"
        )

        print(
            f"  Path: {output_path}"
        )

        print(
            f"  Size: {size_mb:.2f} MB"
        )

        return output_path

    except Exception as e:

        print(
            "\n✗ Float32 conversion failed"
        )

        print(
            f"\nReason:\n{str(e)}"
        )

        return None


# ============================================================
# EXPORT DYNAMIC RANGE QUANTIZED TFLITE
# ============================================================

def export_dynamic_quant(model):

    print("\n" + "=" * 70)
    print("EXPORTING DYNAMIC RANGE QUANTIZED TFLITE")
    print("=" * 70)

    output_path = (
        OUTPUT_DIR /
        "spendsense_dynamic_quant.tflite"
    )

    try:

        converter = (
            tf.lite.TFLiteConverter
            .from_keras_model(model)
        )

        converter.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]

        tflite_model = converter.convert()

        with open(
            output_path,
            "wb"
        ) as f:

            f.write(tflite_model)

        size_mb = (
            output_path.stat().st_size /
            (1024 * 1024)
        )

        print(
            f"✓ Dynamic Quantized TFLite created"
        )

        print(
            f"  Path: {output_path}"
        )

        print(
            f"  Size: {size_mb:.2f} MB"
        )

        return output_path

    except Exception as e:

        print(
            "\n⚠ Dynamic quantization failed"
        )

        print(
            f"\nReason:\n{str(e)}"
        )

        return None


# ============================================================
# EXPORT FLOAT16 TFLITE
# ============================================================

def export_float16(model):

    print("\n" + "=" * 70)
    print("EXPORTING FLOAT16 TFLITE")
    print("=" * 70)

    output_path = (
        OUTPUT_DIR /
        "spendsense_float16.tflite"
    )

    try:

        converter = (
            tf.lite.TFLiteConverter
            .from_keras_model(model)
        )

        converter.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]

        converter.target_spec.supported_types = [
            tf.float16
        ]

        tflite_model = converter.convert()

        with open(
            output_path,
            "wb"
        ) as f:

            f.write(tflite_model)

        size_mb = (
            output_path.stat().st_size /
            (1024 * 1024)
        )

        print(
            f"✓ Float16 TFLite created"
        )

        print(
            f"  Path: {output_path}"
        )

        print(
            f"  Size: {size_mb:.2f} MB"
        )

        return output_path

    except Exception as e:

        print(
            "\n⚠ Float16 conversion failed"
        )

        print(
            f"\nReason:\n{str(e)}"
        )

        return None


# ============================================================
# INSPECT TFLITE MODEL
# ============================================================

def inspect_tflite_model(
    model_path
):

    print("\n" + "=" * 70)
    print(
        f"INSPECTING {model_path.name}"
    )
    print("=" * 70)

    interpreter = tf.lite.Interpreter(

        model_path=str(model_path)
    )

    interpreter.allocate_tensors()

    input_details = (
        interpreter.get_input_details()
    )

    output_details = (
        interpreter.get_output_details()
    )

    print("\nInputs:")

    for detail in input_details:

        print(
            f"\nName: {detail['name']}"
        )

        print(
            f"Shape: {detail['shape']}"
        )

        print(
            f"Dtype: {detail['dtype']}"
        )

    print("\nOutputs:")

    for detail in output_details:

        print(
            f"\nName: {detail['name']}"
        )

        print(
            f"Shape: {detail['shape']}"
        )

        print(
            f"Dtype: {detail['dtype']}"
        )

    return (
        interpreter,
        input_details,
        output_details
    )


# ============================================================
# TEST TFLITE STRING INPUT
# ============================================================

def test_tflite_inference(
    tflite_path,
    sample_text
):

    print("\n" + "=" * 70)
    print(
        f"TESTING TFLITE INFERENCE: "
        f"{tflite_path.name}"
    )
    print("=" * 70)

    try:

        interpreter = tf.lite.Interpreter(
            model_path=str(tflite_path)
        )

        interpreter.allocate_tensors()

        input_details = (
            interpreter.get_input_details()
        )

        output_details = (
            interpreter.get_output_details()
        )

        input_index = (
            input_details[0]["index"]
        )

        input_dtype = (
            input_details[0]["dtype"]
        )

        print(
            f"\nInput dtype: {input_dtype}"
        )

        print(
            f"Input shape: "
            f"{input_details[0]['shape']}"
        )

        # String input handling

        if input_dtype == np.object_:

            input_data = np.array(
                [[sample_text]],
                dtype=object
            )

        elif input_dtype.type is np.bytes_:

            input_data = np.array(
                [[sample_text.encode("utf-8")]],
                dtype=np.bytes_
            )

        else:

            print(
                "\n⚠ Model does not accept "
                "string input."
            )

            return False

        interpreter.resize_tensor_input(

            input_index,

            input_data.shape
        )

        interpreter.allocate_tensors()

        interpreter.set_tensor(

            input_index,

            input_data
        )

        interpreter.invoke()

        outputs = []

        for output in output_details:

            value = interpreter.get_tensor(
                output["index"]
            )

            outputs.append(value)

        print(
            "\n✓ TFLite inference successful"
        )

        for i, output in enumerate(outputs):

            print(
                f"Output {i}: "
                f"shape={output.shape}"
            )

        return True

    except Exception as e:

        print(
            "\n✗ TFLite inference failed"
        )

        print(
            f"\nReason:\n{str(e)}"
        )

        return False


# ============================================================
# COMPARE MODEL FILE SIZES
# ============================================================

def compare_model_sizes():

    print("\n" + "=" * 70)
    print("MODEL SIZE COMPARISON")
    print("=" * 70)

    results = []

    # Keras source

    if MODEL_PATH.exists():

        size = (
            MODEL_PATH.stat().st_size /
            (1024 * 1024)
        )

        results.append({

            "model":
                "Keras",

            "file":
                MODEL_PATH.name,

            "size_mb":
                round(size, 4)
        })

    # TFLite models

    for file in OUTPUT_DIR.glob(
        "*.tflite"
    ):

        size = (
            file.stat().st_size /
            (1024 * 1024)
        )

        results.append({

            "model":
                "TFLite",

            "file":
                file.name,

            "size_mb":
                round(size, 4)
        })

    df = pd.DataFrame(results)

    if len(df) > 0:

        print(
            df.to_string(
                index=False
            )
        )

        df.to_csv(

            OUTPUT_DIR /
            "model_size_comparison.csv",

            index=False
        )

    return results


# ============================================================
# CREATE DEPLOYMENT REPORT
# ============================================================

def create_deployment_report(
    evaluation_results,
    conversion_results,
    labels
):

    print("\n" + "=" * 70)
    print("CREATING DEPLOYMENT REPORT")
    print("=" * 70)

    report = {

        "project":
            "SpendSense",

        "deployment_target":
            "Mobile Application",

        "keras_source_model":
            str(MODEL_PATH),

        "classes": {

            "merchant":
                len(labels["merchant"]),

            "category":
                len(labels["category"])
        },

        "evaluation_results":
            evaluation_results,

        "tflite_models":
            conversion_results
    }

    # --------------------------------------------------------
    # Recommendation
    # --------------------------------------------------------

    successful_models = [

        model

        for model, path
        in conversion_results.items()

        if path is not None
    ]

    if "dynamic_quant" in successful_models:

        recommendation = (
            "Dynamic Range Quantized TFLite "
            "is recommended as the primary "
            "mobile deployment candidate. "
            "Validate prediction consistency "
            "before production deployment."
        )

    elif "float16" in successful_models:

        recommendation = (
            "Float16 TFLite is recommended "
            "for mobile deployment after "
            "device-level validation."
        )

    elif "float32" in successful_models:

        recommendation = (
            "Float32 TFLite successfully "
            "converted and should be used "
            "as the baseline deployment model."
        )

    else:

        recommendation = (
            "Direct TFLite conversion failed. "
            "The model likely requires separating "
            "text preprocessing from the neural "
            "network before TFLite export."
        )

    report[
        "deployment_recommendation"
    ] = recommendation

    report_path = (
        OUTPUT_DIR /
        "deployment_report.json"
    )

    with open(
        report_path,
        "w"
    ) as f:

        json.dump(
            report,
            f,
            indent=4
        )

    print(
        "\nRecommendation:"
    )

    print(
        recommendation
    )

    print(
        f"\n✓ Deployment report saved: "
        f"{report_path}"
    )


# ============================================================
# MAIN
# ============================================================

def main():

    print("\n" + "=" * 70)
    print("SPENDSENSE TFLITE CONVERSION PIPELINE")
    print("=" * 70)

    print(
        f"\nTensorFlow version: "
        f"{tf.__version__}"
    )

    # --------------------------------------------------------
    # Setup
    # --------------------------------------------------------

    setup_directories()

    # --------------------------------------------------------
    # Inspect evaluation
    # --------------------------------------------------------

    (
        evaluation_results,
        prediction_df
    ) = inspect_evaluation()

    # --------------------------------------------------------
    # Load labels
    # --------------------------------------------------------

    labels = load_labels()

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    model = load_keras_model()

    # --------------------------------------------------------
    # Inspect architecture
    # --------------------------------------------------------

    inspect_model(model)

    # --------------------------------------------------------
    # Convert models
    # --------------------------------------------------------

    float32_path = export_float32(
        model
    )

    dynamic_quant_path = export_dynamic_quant(
        model
    )

    float16_path = export_float16(
        model
    )

    conversion_results = {

        "float32":
            str(float32_path)
            if float32_path
            else None,

        "dynamic_quant":
            str(dynamic_quant_path)
            if dynamic_quant_path
            else None,

        "float16":
            str(float16_path)
            if float16_path
            else None
    }

    # --------------------------------------------------------
    # Test converted models
    # --------------------------------------------------------

    sample_text = (
        "icici bank acct xx888 debited "
        "for rs 45.00 on 02-aug-26 "
        "bhupendra dilip credited "
        "upi 621412961083"
    )

    validation_results = {}

    for model_name, path_string in (
        conversion_results.items()
    ):

        if path_string is None:

            validation_results[
                model_name
            ] = False

            continue

        model_path = Path(
            path_string
        )

        try:

            inspect_tflite_model(
                model_path
            )

            success = test_tflite_inference(
                model_path,
                sample_text
            )

            validation_results[
                model_name
            ] = success

        except Exception as e:

            print(
                f"\n⚠ Validation error for "
                f"{model_name}: {e}"
            )

            validation_results[
                model_name
            ] = False

    # --------------------------------------------------------
    # Save validation results
    # --------------------------------------------------------

    validation_path = (
        OUTPUT_DIR /
        "tflite_validation_results.json"
    )

    with open(
        validation_path,
        "w"
    ) as f:

        json.dump(
            validation_results,
            f,
            indent=4
        )

    # --------------------------------------------------------
    # Compare sizes
    # --------------------------------------------------------

    compare_model_sizes()

    # --------------------------------------------------------
    # Deployment report
    # --------------------------------------------------------

    create_deployment_report(

        evaluation_results,

        conversion_results,

        labels
    )

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("TFLITE PIPELINE COMPLETE")
    print("=" * 70)

    print(
        f"\nOutput directory:\n"
        f"{OUTPUT_DIR}"
    )

    print("\nGenerated files:")

    for file in sorted(
        OUTPUT_DIR.rglob("*")
    ):

        if file.is_file():

            size_kb = (
                file.stat().st_size /
                1024
            )

            print(
                f"✓ {file.name} "
                f"({size_kb:.2f} KB)"
            )


if __name__ == "__main__":
    main()

Writing convert_spendsense_tflite.py


In [ ]:
!python convert_spendsense_tflite.py


SPENDSENSE TFLITE CONVERSION PIPELINE

TensorFlow version: 2.20.0
✓ Output directory ready: /content/spendsense_tflite

INSPECTING MODEL EVALUATION RESULTS

Evaluation Results:

MERCHANT
  accuracy: 0.9994
  macro_precision: 0.9980
  macro_recall: 0.9979
  macro_f1: 0.9979
  weighted_f1: 0.9994
CATEGORY
  accuracy: 0.9685
  macro_precision: 0.9652
  macro_recall: 0.9648
  macro_f1: 0.9650
  weighted_f1: 0.9684

✓ Evaluation predictions loaded
Test samples available: 10,765
merchant prediction accuracy from evaluation file: 0.9994
category prediction accuracy from evaluation file: 0.9685

LOADING LABELS
✓ merchant: 734 classes
✓ category: 15 classes

LOADING TRAINED KERAS MODEL
2026-09-02 19:35:23.286212: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788377723.287738   21144 gpu_device.cc:2020] Created device /job:localhost/replica:0

In [ ]:
%%writefile train_spendsense_v3_final.py

"""
================================================================
SpendSense V3 - Final Production Training Pipeline
FIXED: Merchant-aware train/validation/test split
================================================================

Fixes:
- Prevents unseen merchant labels in validation/test
- Every merchant evaluated has training examples
- Vocabulary built only from training text
- Integer input architecture for TFLite compatibility
- Multi-output CNN:
    Merchant Classification
    Category Classification
- TFLite export + validation
================================================================
"""

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

DATA_PATH = "/content/indian_payment_transaction_dataset (1).csv"

BASE_DIR = Path("/content/spendsense_final")

MODEL_DIR = BASE_DIR / "models"
ARTIFACT_DIR = BASE_DIR / "artifacts"
EVALUATION_DIR = BASE_DIR / "evaluation"
PLOTS_DIR = EVALUATION_DIR / "plots"
VALIDATION_DIR = BASE_DIR / "validation"

VALIDATION_SIZE = 0.10
TEST_SIZE = 0.10

MAX_SEQUENCE_LENGTH = 128

EMBEDDING_DIM = 64
CNN_FILTERS = 128
DENSE_UNITS = 128
DROPOUT_RATE = 0.30

BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 0.001

TOP_CLASSES_CONFUSION_MATRIX = 30

# Merchant splitting rules
MIN_SAMPLES_FOR_EVALUATION = 3


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    tf.random.set_seed(seed)

    print(f"✓ Random seed set: {seed}")


# ============================================================
# DIRECTORIES
# ============================================================

def create_directories():

    directories = [
        BASE_DIR,
        MODEL_DIR,
        ARTIFACT_DIR,
        EVALUATION_DIR,
        PLOTS_DIR,
        VALIDATION_DIR,
    ]

    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)

    print("✓ Output directories ready")


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text)
    text = text.lower().strip()

    # Normalize repeated whitespace
    text = " ".join(text.split())

    return text


# ============================================================
# LOAD DATA
# ============================================================

def load_dataset():

    print("\n" + "=" * 70)
    print("LOADING DATASET")
    print("=" * 70)

    if not os.path.exists(DATA_PATH):

        print(f"\nDataset not found: {DATA_PATH}")
        print("\nFiles in /content:")

        for file in os.listdir("/content"):
            print(" -", file)

        raise FileNotFoundError(DATA_PATH)

    df = pd.read_csv(DATA_PATH)

    print(f"\nOriginal samples: {len(df):,}")
    print(f"Columns: {df.columns.tolist()}")

    required_columns = [
        "transaction_text",
        "merchant",
        "category",
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    df = df[required_columns].copy()

    df["transaction_text"] = (
        df["transaction_text"]
        .apply(normalize_text)
    )

    df["merchant"] = (
        df["merchant"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

    df["category"] = (
        df["category"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )

    # Remove empty transaction text
    before = len(df)

    df = df[
        df["transaction_text"].str.len() > 0
    ].copy()

    print(
        f"Removed empty texts: "
        f"{before - len(df)}"
    )

    # Remove exact duplicates
    before = len(df)

    df = df.drop_duplicates().copy()

    print(
        f"Removed duplicates: "
        f"{before - len(df)}"
    )

    # Shuffle
    df = df.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    print(f"\nFinal samples: {len(df):,}")
    print(f"Unique merchants: {df['merchant'].nunique()}")
    print(f"Unique categories: {df['category'].nunique()}")

    return df


# ============================================================
# MERCHANT-AWARE SPLIT
# ============================================================

def merchant_aware_split(df):

    """
    Split each merchant individually.

    Guarantees:
    - Every merchant in validation exists in training
    - Every merchant in test exists in training
    - Rare merchants remain entirely in training

    Example:

    Merchant A = 300 samples
        240 train
         30 validation
         30 test

    Merchant B = 2 samples
          2 train
          0 validation
          0 test
    """

    print("\n" + "=" * 70)
    print("MERCHANT-AWARE DATASET SPLIT")
    print("=" * 70)

    train_parts = []
    val_parts = []
    test_parts = []

    merchant_counts = (
        df["merchant"]
        .value_counts()
    )

    rare_merchants = []
    split_merchants = []

    rng = np.random.RandomState(SEED)

    for merchant, count in merchant_counts.items():

        merchant_df = df[
            df["merchant"] == merchant
        ].copy()

        # Shuffle merchant examples
        merchant_df = merchant_df.sample(
            frac=1,
            random_state=int(
                rng.randint(0, 1_000_000)
            )
        )

        # ----------------------------------------------------
        # Rare merchant
        # ----------------------------------------------------

        if count < MIN_SAMPLES_FOR_EVALUATION:

            train_parts.append(merchant_df)

            rare_merchants.append(
                merchant
            )

            continue

        # ----------------------------------------------------
        # Determine split sizes
        # ----------------------------------------------------

        n_test = max(
            1,
            int(round(count * TEST_SIZE))
        )

        n_val = max(
            1,
            int(round(count * VALIDATION_SIZE))
        )

        # Guarantee at least 1 training sample
        max_eval_samples = count - 1

        if (
            n_test + n_val
            >
            max_eval_samples
        ):

            overflow = (
                n_test + n_val
                -
                max_eval_samples
            )

            # Reduce validation first
            n_val = max(
                0,
                n_val - overflow
            )

            overflow = (
                n_test + n_val
                -
                max_eval_samples
            )

            if overflow > 0:

                n_test = max(
                    0,
                    n_test - overflow
                )

        # ----------------------------------------------------
        # Final split
        # ----------------------------------------------------

        test_part = merchant_df.iloc[
            :n_test
        ]

        val_part = merchant_df.iloc[
            n_test:
            n_test + n_val
        ]

        train_part = merchant_df.iloc[
            n_test + n_val:
        ]

        # Safety guarantee
        if len(train_part) == 0:

            train_part = merchant_df.iloc[:1]

            remaining = merchant_df.iloc[1:]

            split_point = len(remaining) // 2

            val_part = remaining.iloc[
                :split_point
            ]

            test_part = remaining.iloc[
                split_point:
            ]

        train_parts.append(train_part)

        if len(val_part) > 0:
            val_parts.append(val_part)

        if len(test_part) > 0:
            test_parts.append(test_part)

        split_merchants.append(
            merchant
        )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    train_df = pd.concat(
        train_parts,
        ignore_index=True
    )

    val_df = pd.concat(
        val_parts,
        ignore_index=True
    )

    test_df = pd.concat(
        test_parts,
        ignore_index=True
    )

    # Final shuffle
    train_df = train_df.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    val_df = val_df.sample(
        frac=1,
        random_state=SEED + 1
    ).reset_index(drop=True)

    test_df = test_df.sample(
        frac=1,
        random_state=SEED + 2
    ).reset_index(drop=True)

    print(f"\nTrain samples: {len(train_df):,}")
    print(f"Validation samples: {len(val_df):,}")
    print(f"Test samples: {len(test_df):,}")

    print(
        f"\nRare merchants kept "
        f"only in training: "
        f"{len(rare_merchants)}"
    )

    print(
        f"Merchants split across sets: "
        f"{len(split_merchants)}"
    )

    # --------------------------------------------------------
    # VALIDATION CHECK
    # --------------------------------------------------------

    train_merchants = set(
        train_df["merchant"]
    )

    val_merchants = set(
        val_df["merchant"]
    )

    test_merchants = set(
        test_df["merchant"]
    )

    unseen_val = (
        val_merchants
        -
        train_merchants
    )

    unseen_test = (
        test_merchants
        -
        train_merchants
    )

    if unseen_val:
        raise RuntimeError(
            f"Unseen validation merchants: "
            f"{unseen_val}"
        )

    if unseen_test:
        raise RuntimeError(
            f"Unseen test merchants: "
            f"{unseen_test}"
        )

    print(
        "\n✓ Merchant split validation PASSED"
    )

    # --------------------------------------------------------
    # Category statistics
    # --------------------------------------------------------

    print("\nCategory distribution:")

    for name, split_df in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df),
    ]:

        print(
            f"\n{name}:"
        )

        print(
            split_df["category"]
            .value_counts(normalize=True)
            .head(10)
            .round(4)
            .to_string()
        )

    return train_df, val_df, test_df


# ============================================================
# BUILD VOCABULARY
# ============================================================

def build_vocabulary(train_texts):

    print("\n" + "=" * 70)
    print("BUILDING VOCABULARY")
    print("=" * 70)

    characters = set()

    for text in train_texts:
        characters.update(list(text))

    characters = sorted(characters)

    vocabulary = {
        "<PAD>": 0,
        "<UNK>": 1,
    }

    for index, character in enumerate(
        characters,
        start=2
    ):
        vocabulary[character] = index

    path = (
        ARTIFACT_DIR /
        "vocabulary.json"
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            vocabulary,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"✓ Vocabulary size: "
        f"{len(vocabulary)}"
    )

    return vocabulary


# ============================================================
# TEXT TO SEQUENCE
# ============================================================

def text_to_sequence(
    text,
    vocabulary
):

    text = normalize_text(text)

    unk_id = vocabulary["<UNK>"]
    pad_id = vocabulary["<PAD>"]

    sequence = [
        vocabulary.get(char, unk_id)
        for char in text
    ]

    # Truncate
    sequence = sequence[
        :MAX_SEQUENCE_LENGTH
    ]

    # Post-padding
    if len(sequence) < MAX_SEQUENCE_LENGTH:

        sequence += (
            [pad_id] *
            (
                MAX_SEQUENCE_LENGTH
                -
                len(sequence)
            )
        )

    return sequence


def create_sequences(
    texts,
    vocabulary
):

    print(
        f"Tokenizing "
        f"{len(texts):,} samples..."
    )

    sequences = np.array(
        [
            text_to_sequence(
                text,
                vocabulary
            )
            for text in texts
        ],
        dtype=np.int32
    )

    return sequences


# ============================================================
# LABEL ENCODERS
# ============================================================

def create_label_encoders(
    train_df
):

    print("\n" + "=" * 70)
    print("CREATING LABEL ENCODERS")
    print("=" * 70)

    encoders = {}

    for task in [
        "merchant",
        "category",
    ]:

        encoder = LabelEncoder()

        encoder.fit(
            train_df[task]
            .astype(str)
        )

        encoders[task] = encoder

        labels = (
            encoder.classes_
            .tolist()
        )

        path = (
            ARTIFACT_DIR /
            f"{task}_labels.json"
        )

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                labels,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"✓ {task}: "
            f"{len(labels)} classes"
        )

    return encoders


# ============================================================
# VALIDATE LABELS
# ============================================================

def validate_all_labels(
    train_df,
    val_df,
    test_df,
    encoders
):

    print("\n" + "=" * 70)
    print("LABEL VALIDATION")
    print("=" * 70)

    for task in [
        "merchant",
        "category",
    ]:

        known = set(
            encoders[task].classes_
        )

        for split_name, split_df in [
            ("validation", val_df),
            ("test", test_df),
        ]:

            actual = set(
                split_df[task]
                .astype(str)
            )

            unknown = actual - known

            if unknown:

                raise RuntimeError(
                    f"Unknown {task} labels "
                    f"in {split_name}: "
                    f"{list(unknown)[:20]}"
                )

            print(
                f"✓ {task} labels valid "
                f"for {split_name}"
            )


# ============================================================
# PREPARE TARGETS
# ============================================================

def prepare_targets(
    df,
    encoders
):

    targets = {}

    for task in [
        "merchant",
        "category",
    ]:

        targets[task] = (
            encoders[task]
            .transform(
                df[task].astype(str)
            )
            .astype(np.int32)
        )

    return targets


# ============================================================
# CREATE TF DATASET
# ============================================================

def create_tf_dataset(
    sequences,
    targets,
    training=False
):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            sequences,
            {
                "merchant":
                    targets["merchant"],
                "category":
                    targets["category"],
            }
        )
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=min(
                len(sequences),
                20000
            ),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(
        BATCH_SIZE
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


# ============================================================
# BUILD MODEL
# ============================================================

def build_model(
    vocabulary,
    encoders
):

    print("\n" + "=" * 70)
    print("BUILDING MODEL")
    print("=" * 70)

    vocabulary_size = len(vocabulary)

    merchant_classes = len(
        encoders["merchant"].classes_
    )

    category_classes = len(
        encoders["category"].classes_
    )

    inputs = tf.keras.Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype=tf.int32,
        name="token_ids"
    )

    x = tf.keras.layers.Embedding(
        input_dim=vocabulary_size,
        output_dim=EMBEDDING_DIM,
        name="embedding"
    )(inputs)

    branches = []

    for kernel_size in [3, 5, 7]:

        branch = tf.keras.layers.Conv1D(
            filters=CNN_FILTERS,
            kernel_size=kernel_size,
            padding="same",
            activation="relu",
            name=f"conv_{kernel_size}"
        )(x)

        branch = tf.keras.layers.GlobalMaxPooling1D(
            name=f"pool_{kernel_size}"
        )(branch)

        branches.append(branch)

    x = tf.keras.layers.Concatenate(
        name="cnn_features"
    )(branches)

    x = tf.keras.layers.Dense(
        DENSE_UNITS,
        activation="relu",
        name="shared_dense"
    )(x)

    x = tf.keras.layers.Dropout(
        DROPOUT_RATE,
        name="dropout"
    )(x)

    merchant_output = tf.keras.layers.Dense(
        merchant_classes,
        activation="softmax",
        name="merchant"
    )(x)

    category_output = tf.keras.layers.Dense(
        category_classes,
        activation="softmax",
        name="category"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            "merchant": merchant_output,
            "category": category_output,
        },
        name="SpendSenseV3"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss={
            "merchant":
                "sparse_categorical_crossentropy",
            "category":
                "sparse_categorical_crossentropy",
        },

        loss_weights={
            "merchant": 0.6,
            "category": 1.0,
        },

        metrics={
            "merchant": ["accuracy"],
            "category": ["accuracy"],
        }
    )

    model.summary()

    return model


# ============================================================
# TRAIN
# ============================================================

def train_model(
    model,
    train_dataset,
    val_dataset
):

    print("\n" + "=" * 70)
    print("TRAINING")
    print("=" * 70)

    checkpoint_path = (
        MODEL_DIR /
        "best_spendsense_v3.keras"
    )

    callbacks = [

        tf.keras.callbacks.EarlyStopping(
            monitor="val_category_accuracy",
            mode="max",
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_category_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        ),
    ]

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    history_data = {
        key: [
            float(v)
            for v in values
        ]
        for key, values
        in history.history.items()
    }

    with open(
        EVALUATION_DIR /
        "training_history.json",
        "w"
    ) as f:
        json.dump(
            history_data,
            f,
            indent=2
        )

    return history


# ============================================================
# TRAINING PLOTS
# ============================================================

def save_training_plots():

    history_path = (
        EVALUATION_DIR /
        "training_history.json"
    )

    if not history_path.exists():
        return

    with open(history_path) as f:
        history = json.load(f)

    # Loss
    if (
        "loss" in history
        and "val_loss" in history
    ):

        plt.figure(figsize=(10, 6))

        plt.plot(
            history["loss"],
            label="Training Loss"
        )

        plt.plot(
            history["val_loss"],
            label="Validation Loss"
        )

        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training vs Validation Loss")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()

        plt.savefig(
            PLOTS_DIR /
            "training_loss.png",
            dpi=200
        )

        plt.close()

    # Category accuracy
    if (
        "category_accuracy" in history
        and "val_category_accuracy" in history
    ):

        plt.figure(figsize=(10, 6))

        plt.plot(
            history["category_accuracy"],
            label="Training Accuracy"
        )

        plt.plot(
            history["val_category_accuracy"],
            label="Validation Accuracy"
        )

        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title("Category Accuracy")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()

        plt.savefig(
            PLOTS_DIR /
            "category_accuracy.png",
            dpi=200
        )

        plt.close()


# ============================================================
# EVALUATION
# ============================================================

def evaluate_model(
    model,
    test_sequences,
    test_df,
    test_targets,
    encoders
):

    print("\n" + "=" * 70)
    print("FINAL EVALUATION")
    print("=" * 70)

    predictions = model.predict(
        test_sequences,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    metrics = {}

    output_df = test_df.copy()

    for task in [
        "merchant",
        "category",
    ]:

        probabilities = predictions[task]

        y_pred = np.argmax(
            probabilities,
            axis=1
        )

        confidence = np.max(
            probabilities,
            axis=1
        )

        y_true = test_targets[task]

        task_metrics = {

            "accuracy":
                float(
                    accuracy_score(
                        y_true,
                        y_pred
                    )
                ),

            "macro_precision":
                float(
                    precision_score(
                        y_true,
                        y_pred,
                        average="macro",
                        zero_division=0
                    )
                ),

            "macro_recall":
                float(
                    recall_score(
                        y_true,
                        y_pred,
                        average="macro",
                        zero_division=0
                    )
                ),

            "macro_f1":
                float(
                    f1_score(
                        y_true,
                        y_pred,
                        average="macro",
                        zero_division=0
                    )
                ),

            "weighted_f1":
                float(
                    f1_score(
                        y_true,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                ),
        }

        metrics[task] = task_metrics

        print(f"\n{task.upper()}")

        for key, value in task_metrics.items():
            print(f"{key}: {value:.4f}")

        # Decode
        true_labels = [
            encoders[task].classes_[i]
            for i in y_true
        ]

        predicted_labels = [
            encoders[task].classes_[i]
            for i in y_pred
        ]

        output_df[f"true_{task}"] = true_labels
        output_df[f"predicted_{task}"] = predicted_labels
        output_df[f"{task}_confidence"] = confidence

        output_df[f"{task}_correct"] = (
            np.array(true_labels)
            ==
            np.array(predicted_labels)
        )

        # Classification report
        report = classification_report(
            y_true,
            y_pred,
            labels=np.arange(
                len(encoders[task].classes_)
            ),
            target_names=encoders[task].classes_,
            output_dict=True,
            zero_division=0
        )

        pd.DataFrame(report).transpose().to_csv(
            EVALUATION_DIR /
            f"{task}_classification_report.csv"
        )

        # Confusion matrix
        cm = confusion_matrix(
            y_true,
            y_pred
        )

        pd.DataFrame(cm).to_csv(
            EVALUATION_DIR /
            f"{task}_confusion_matrix.csv",
            index=False
        )

    # Save predictions
    output_df.to_csv(
        EVALUATION_DIR /
        "all_test_predictions.csv",
        index=False
    )

    incorrect = output_df[
        (~output_df["merchant_correct"])
        |
        (~output_df["category_correct"])
    ]

    incorrect.to_csv(
        EVALUATION_DIR /
        "incorrect_predictions.csv",
        index=False
    )

    with open(
        EVALUATION_DIR /
        "evaluation_results.json",
        "w"
    ) as f:
        json.dump(
            metrics,
            f,
            indent=4
        )

    return metrics, predictions


# ============================================================
# TFLITE EXPORT
# ============================================================

def export_tflite_models(model):

    print("\n" + "=" * 70)
    print("EXPORTING TFLITE")
    print("=" * 70)

    exported = {}

    configs = {

        "float32": {
            "filename":
                "spendsense_float32.tflite",
            "optimization":
                None,
        },

        "dynamic_quant": {
            "filename":
                "spendsense_dynamic_quant.tflite",
            "optimization":
                "dynamic",
        },

        "float16": {
            "filename":
                "spendsense_float16.tflite",
            "optimization":
                "float16",
        },
    }

    for name, config in configs.items():

        try:

            print(
                f"\nExporting {name}..."
            )

            converter = (
                tf.lite.TFLiteConverter
                .from_keras_model(model)
            )

            if config["optimization"] == "dynamic":

                converter.optimizations = [
                    tf.lite.Optimize.DEFAULT
                ]

            elif config["optimization"] == "float16":

                converter.optimizations = [
                    tf.lite.Optimize.DEFAULT
                ]

                converter.target_spec.supported_types = [
                    tf.float16
                ]

            tflite_model = converter.convert()

            path = (
                MODEL_DIR /
                config["filename"]
            )

            with open(path, "wb") as f:
                f.write(tflite_model)

            size_mb = (
                path.stat().st_size /
                (1024 * 1024)
            )

            print(
                f"✓ Exported: "
                f"{path.name}"
            )

            print(
                f"  Size: "
                f"{size_mb:.3f} MB"
            )

            exported[name] = path

        except Exception as e:

            print(
                f"⚠ Failed {name}: {e}"
            )

    return exported


# ============================================================
# TFLITE VALIDATION
# ============================================================

def validate_tflite(
    path,
    test_sequences,
    keras_predictions
):

    interpreter = tf.lite.Interpreter(
        model_path=str(path)
    )

    interpreter.allocate_tensors()

    input_details = (
        interpreter.get_input_details()
    )

    output_details = (
        interpreter.get_output_details()
    )

    keras_merchant = np.argmax(
        keras_predictions["merchant"],
        axis=1
    )

    keras_category = np.argmax(
        keras_predictions["category"],
        axis=1
    )

    merchant_dim = (
        keras_predictions["merchant"]
        .shape[-1]
    )

    category_dim = (
        keras_predictions["category"]
        .shape[-1]
    )

    tflite_merchant = []
    tflite_category = []

    for sequence in test_sequences:

        input_data = np.expand_dims(
            sequence,
            axis=0
        ).astype(np.int32)

        interpreter.set_tensor(
            input_details[0]["index"],
            input_data
        )

        interpreter.invoke()

        outputs = [
            interpreter.get_tensor(
                detail["index"]
            )
            for detail in output_details
        ]

        merchant_output = None
        category_output = None

        for output in outputs:

            if output.shape[-1] == merchant_dim:
                merchant_output = output

            elif output.shape[-1] == category_dim:
                category_output = output

        if merchant_output is None:
            merchant_output = outputs[0]

        if category_output is None:
            category_output = outputs[1]

        tflite_merchant.append(
            np.argmax(
                merchant_output[0]
            )
        )

        tflite_category.append(
            np.argmax(
                category_output[0]
            )
        )

    merchant_agreement = float(
        np.mean(
            keras_merchant
            ==
            np.array(tflite_merchant)
        )
    )

    category_agreement = float(
        np.mean(
            keras_category
            ==
            np.array(tflite_category)
        )
    )

    return {

        "model": path.name,

        "size_mb": float(
            path.stat().st_size /
            (1024 * 1024)
        ),

        "merchant_agreement":
            merchant_agreement,

        "category_agreement":
            category_agreement,

        "status":
            "validated",
    }


# ============================================================
# SAVE CONFIGURATION
# ============================================================

def save_artifacts(
    vocabulary,
    encoders,
    metrics,
    validation_results
):

    tokenizer_config = {

        "type":
            "character_level",

        "normalization":
            "lowercase_strip_normalize_spaces",

        "max_sequence_length":
            MAX_SEQUENCE_LENGTH,

        "padding":
            "post",

        "truncation":
            "post",

        "pad_token":
            "<PAD>",

        "pad_token_id":
            0,

        "unknown_token":
            "<UNK>",

        "unknown_token_id":
            1,
    }

    with open(
        ARTIFACT_DIR /
        "tokenizer_config.json",
        "w"
    ) as f:

        json.dump(
            tokenizer_config,
            f,
            indent=4
        )

    model_config = {

        "project":
            "SpendSense",

        "version":
            "V3 Production",

        "input": {

            "shape":
                [1, MAX_SEQUENCE_LENGTH],

            "dtype":
                "int32",
        },

        "architecture": {

            "embedding_dim":
                EMBEDDING_DIM,

            "cnn_filters":
                CNN_FILTERS,

            "kernel_sizes":
                [3, 5, 7],

            "dense_units":
                DENSE_UNITS,

            "dropout":
                DROPOUT_RATE,
        },

        "vocabulary_size":
            len(vocabulary),

        "merchant_classes":
            len(
                encoders["merchant"].classes_
            ),

        "category_classes":
            len(
                encoders["category"].classes_
            ),

        "keras_evaluation":
            metrics,

        "tflite_validation":
            validation_results,
    }

    with open(
        ARTIFACT_DIR /
        "model_config.json",
        "w"
    ) as f:

        json.dump(
            model_config,
            f,
            indent=4
        )


# ============================================================
# README
# ============================================================

def create_readme():

    content = f"""
============================================================
SPENDSENSE V3 - MOBILE DEPLOYMENT
============================================================

INPUT
-----

Shape:
[1, {MAX_SEQUENCE_LENGTH}]

Type:
int32


TOKENIZATION
------------

1. Lowercase transaction text
2. Strip leading/trailing whitespace
3. Normalize multiple spaces
4. Convert each character using vocabulary.json
5. Unknown characters = ID 1
6. PAD = ID 0
7. Truncate to {MAX_SEQUENCE_LENGTH}
8. Post-pad to {MAX_SEQUENCE_LENGTH}


OUTPUT
------

Output heads:

1. merchant
2. category


DECODE LABELS
-------------

Merchant:
artifacts/merchant_labels.json

Category:
artifacts/category_labels.json


IMPORTANT
---------

Flutter MUST use the exact vocabulary.json.

Do NOT generate vocabulary dynamically.

============================================================
"""

    with open(
        BASE_DIR /
        "README_DEPLOYMENT.txt",
        "w"
    ) as f:
        f.write(content)


# ============================================================
# MAIN
# ============================================================

def main():

    print("\n" + "=" * 70)
    print("SPENDSENSE V3 FINAL PIPELINE")
    print("=" * 70)

    print(
        f"TensorFlow Version: "
        f"{tf.__version__}"
    )

    set_seed()

    create_directories()

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    df = load_dataset()

    # --------------------------------------------------------
    # Merchant-aware split
    # --------------------------------------------------------

    train_df, val_df, test_df = (
        merchant_aware_split(df)
    )

    # --------------------------------------------------------
    # Vocabulary from training ONLY
    # --------------------------------------------------------

    vocabulary = build_vocabulary(
        train_df[
            "transaction_text"
        ].tolist()
    )

    # --------------------------------------------------------
    # Encoders from training ONLY
    # --------------------------------------------------------

    encoders = create_label_encoders(
        train_df
    )

    # --------------------------------------------------------
    # Critical validation
    # --------------------------------------------------------

    validate_all_labels(
        train_df,
        val_df,
        test_df,
        encoders
    )

    # --------------------------------------------------------
    # Sequences
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("CREATING INTEGER SEQUENCES")
    print("=" * 70)

    train_sequences = create_sequences(
        train_df[
            "transaction_text"
        ].tolist(),
        vocabulary
    )

    val_sequences = create_sequences(
        val_df[
            "transaction_text"
        ].tolist(),
        vocabulary
    )

    test_sequences = create_sequences(
        test_df[
            "transaction_text"
        ].tolist(),
        vocabulary
    )

    print(
        f"\nTrain shape: "
        f"{train_sequences.shape}"
    )

    print(
        f"Validation shape: "
        f"{val_sequences.shape}"
    )

    print(
        f"Test shape: "
        f"{test_sequences.shape}"
    )

    # --------------------------------------------------------
    # Targets
    # --------------------------------------------------------

    train_targets = prepare_targets(
        train_df,
        encoders
    )

    val_targets = prepare_targets(
        val_df,
        encoders
    )

    test_targets = prepare_targets(
        test_df,
        encoders
    )

    # --------------------------------------------------------
    # TF datasets
    # --------------------------------------------------------

    train_dataset = create_tf_dataset(
        train_sequences,
        train_targets,
        training=True
    )

    val_dataset = create_tf_dataset(
        val_sequences,
        val_targets,
        training=False
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = build_model(
        vocabulary,
        encoders
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train_model(
        model,
        train_dataset,
        val_dataset
    )

    # --------------------------------------------------------
    # Save Keras
    # --------------------------------------------------------

    keras_path = (
        MODEL_DIR /
        "spendsense_v3.keras"
    )

    model.save(keras_path)

    print(
        f"\n✓ Keras saved: "
        f"{keras_path}"
    )

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------

    save_training_plots()

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    metrics, keras_predictions = (
        evaluate_model(
            model,
            test_sequences,
            test_df,
            test_targets,
            encoders
        )
    )

    # --------------------------------------------------------
    # TFLite export
    # --------------------------------------------------------

    tflite_models = export_tflite_models(
        model
    )

    # --------------------------------------------------------
    # TFLite validation
    # --------------------------------------------------------

    validation_results = {}

    for name, path in tflite_models.items():

        try:

            print(
                f"\nValidating "
                f"{name}..."
            )

            result = validate_tflite(
                path,
                test_sequences,
                keras_predictions
            )

            validation_results[name] = result

            print(result)

        except Exception as e:

            print(
                f"⚠ Validation failed "
                f"for {name}: {e}"
            )

            validation_results[name] = {

                "status":
                    "failed",

                "error":
                    str(e),
            }

    with open(
        VALIDATION_DIR /
        "tflite_validation_results.json",
        "w"
    ) as f:

        json.dump(
            validation_results,
            f,
            indent=4
        )

    # --------------------------------------------------------
    # Final artifacts
    # --------------------------------------------------------

    save_artifacts(
        vocabulary,
        encoders,
        metrics,
        validation_results
    )

    create_readme()

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary = {

        "version":
            "SpendSense V3",

        "dataset_size":
            len(df),

        "train_size":
            len(train_df),

        "validation_size":
            len(val_df),

        "test_size":
            len(test_df),

        "unique_merchants":
            len(
                encoders[
                    "merchant"
                ].classes_
            ),

        "unique_categories":
            len(
                encoders[
                    "category"
                ].classes_
            ),

        "metrics":
            metrics,

        "tflite":
            validation_results,
    }

    with open(
        BASE_DIR /
        "final_summary.json",
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=4
        )

    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE")
    print("=" * 70)

    print(
        f"\nFinal directory:\n"
        f"{BASE_DIR}"
    )

    print("\nFiles:")

    for file in sorted(
        BASE_DIR.rglob("*")
    ):

        if file.is_file():

            size = (
                file.stat().st_size /
                1024
            )

            print(
                f"✓ "
                f"{file.relative_to(BASE_DIR)} "
                f"({size:.1f} KB)"
            )


if __name__ == "__main__":
    main()

Writing train_spendsense_v3_final.py


In [ ]:
!python train_spendsense_v3_final.py


SPENDSENSE V3 FINAL PIPELINE
TensorFlow Version: 2.20.0
✓ Random seed set: 42
✓ Output directories ready

LOADING DATASET

Original samples: 133,131
Columns: ['transaction_text', 'merchant', 'category', 'category_id', 'augmentation_type']
Removed empty texts: 0
Removed duplicates: 25676

Final samples: 107,455
Unique merchants: 734
Unique categories: 15

MERCHANT-AWARE DATASET SPLIT

Train samples: 85,925
Validation samples: 10,765
Test samples: 10,765

Rare merchants kept only in training: 9
Merchants split across sets: 725

✓ Merchant split validation PASSED

Category distribution:

Train:
category
Personal        0.0978
Shopping        0.0972
Food            0.0934
Utilities       0.0680
Subscription    0.0670
Education       0.0658
Medical         0.0645
Groceries       0.0628
Travel          0.0623
Transport       0.0616

Validation:
category
Personal        0.0989
Shopping        0.0967
Food            0.0935
Utilities       0.0689
Subscription    0.0662
Education       0.0658
M

In [ ]:
import shutil
from google.colab import files

# 1. Zip the directory (Replace 'your_folder_name' with your actual folder name)
# This creates a file named 'your_folder_name.zip' in the /content directory
shutil.make_archive('spendsense_final', 'zip', '/content/spendsense_final')

# 2. Trigger the browser download
files.download('spendsense_final.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>